## EDA on supply chain data to see what should be cleaned

In [0]:
%python
df = spark.sql("SELECT * FROM supply_chain_demo_cat.bronze.raw_supply_chain")

display(df)

In [0]:
%python
df_metadata = spark.sql("FROM supply_chain_demo_cat.bronze.metadata")
df_metadata.display()

In [0]:
%python
# EDA

df_metadata.count(), len(df.columns)

In [0]:
%python
print(df.columns)

In [0]:
%python
# Comparing both df with symetric difference
# Idea is to do an union and to take the whole thingh minus the intersection content.
set(df_metadata.select("FIELDS").toPandas()["FIELDS"]).symmetric_difference(
	set(df.columns)
)

In [0]:
%python
df.select("Order Zipcode").limit(3).display()

In [0]:
%python
# Same code as per above nbut using .distinct to see diff values
df.select("Order Zipcode").distinct().display()

In [0]:
%python
df.select("Order Zipcode").distinct().count()

In [0]:
%python
# Number of rows and columns
print(f"Number of rows {df.count()}")
print(f"Number of columns {len(df.columns)}")

## Explore for cleaning columns

In [0]:
%python
df.printSchema()


In [0]:
%python
df.display()

In [0]:
%python
# We want to see the diff types
df.select("Type").distinct().display()

In [0]:
%python
# Like using .describe() in Pandas
df.summary().display()

In [0]:
%python
# best way
df.select(
[
	column
	for column, type_ in df.dtypes
	if type_ in ("int","bigint","double","decimal")
]
).summary().display()


In [0]:
%python
# As an ex, checking what "Benefit per order" means bu reading in the result
df_metadata.display()

In [0]:
%python
df.select("Customer Country").distinct().display()

df.select("Order Country").distinct().display()
# In the 2nd table, we will see some issues in few names, we need the encoding latin and SHOULD be done in the BRONZE layer when we ingest the data

In [0]:
%python
# calculating the nulls
df.select("Customer Zipcode").filter(df["Customer Zipcode"].isNull()).count()

In [0]:
%python
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns])

null_counts = null_counts.collect()[0].asDict()
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

In [0]:
%python
df.select("shipping date (DateOrders)").display()

# Need to change go timestamp to be abel to do calculations


In [0]:
%python
df.select("Product Description").distinct().display()


In [0]:
%python
import re

def to_snake_case(name):
	return re.sub(r"[\s]+", "_", name.strip().casefold())

def rename_columns_to_snake_case(df):
	new_columns = [to_snake_case(column) for column in df.columns]
	return df.toDF(*new_columns)

df_column_alias = rename_columns_to_snake_case(df)

df_column_alias.limit(5).display()

In [0]:
%python
# chang the timestamp

from pyspark.sql.functions import to_timestamp, col
df_timestamp = df_column_alias.withColumn(
	"shipping_date", to_timestamp(col("shipping_date_(dateorders)"), "M/d/yyyy H:mm")
)

df_timestamp.select("shipping_date").limit(3).display()

In [0]:
%python
from pyspark.sql.functions import coalesce, lit, when

df_cleaned = (
df_timestamp.withColumn(
    "customer_lname", coalesce("customer_lname", lit("-"))
).withColumn(
"customer_zipcode", coalesce(col("customer_zipcode").cast("string"), lit("unknown"))
).withColumn("order_zipcode", coalesce(col("order_zipcode").cast("string"), lit("unknown"))
).withColumn(
"customer_country",
when(col("customer_country") == "EE. UU.", "United States").otherwise(col("customer_country"))
)
).drop("product_description","customer_email","customer_password")

#display(df_cleaned.select("customer_country","customer_zipcode").distinct())

display(df_cleaned)

In [0]:
%python
df_cleaned.display()

## Above eda is not finished (video stops here) and now pipeline